# Counterfactual Generation: Progressive Experiment Comparison

This notebook compares all experiments from the progressive counterfactual generation pipeline:

| Exp | Description | SCM Type | Features | Batch |
|-----|-------------|----------|----------|-------|
| 0 | Linear sanity check | 1 fixed, linear | 2 | 1 |
| 1 | Single nonlinear SCM | 1 fixed, Tanh | 3 | 1 |
| 2A | Feature scaling (5f) | 1 fixed, Tanh | 5 | 1 |
| 2B | Feature scaling (10f) | 1 fixed, Tanh | 10 | 1 |
| 3 | SCM family (ICL) | 10 fixed, Tanh | 5 | 4 |
| 4 | Diverse SCMs | Random per batch | 5 | 4 |

In [ ]:
import json
import os
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Load all experiment results
results_dir = os.path.join(os.path.dirname(os.getcwd()), 'docs', 'results')
if not os.path.exists(results_dir):
    results_dir = '../docs/results'

experiments = {}
exp_dirs = {
    'Exp 0': 'exp0',
    'Exp 1': 'exp1_fixed',
    'Exp 2A': 'exp2a',
    'Exp 2B': 'exp2b',
    'Exp 3': 'exp3',
    'Exp 4': 'exp4',
}

for name, dirname in exp_dirs.items():
    path = os.path.join(results_dir, dirname, 'results.json')
    if os.path.exists(path):
        with open(path) as f:
            experiments[name] = json.load(f)
        print(f"Loaded {name} from {dirname}/")
    else:
        print(f"WARNING: {path} not found")

print(f"\nLoaded {len(experiments)} experiments")

## 1. Summary Table

Key metrics across all experiments, with pass/fail thresholds from the plan.

In [ ]:
# Build summary table
headers = ['Experiment', 'Features', 'Delta MSE', 'Sign Acc.', 'SCM Validity',
           'Loss Reduction', 'Train Time (s)', 'Result']

# Success criteria from the plan
criteria = {
    'Exp 0':  {'delta_mse': 0.01, 'scm_validity': 0.95},
    'Exp 1':  {'delta_mse': 0.1, 'sign_accuracy': 0.85, 'scm_validity': 0.70, 'loss_reduction': 0.80},
    'Exp 2A': {'delta_mse': 0.15, 'sign_accuracy': 0.80, 'scm_validity': 0.60},
    'Exp 2B': {'delta_mse': 0.25, 'sign_accuracy': 0.70, 'scm_validity': 0.50},
    'Exp 3':  {'delta_mse': 0.3, 'scm_validity': 0.50},
    'Exp 4':  {'delta_mse': 1.0, 'scm_validity': 0.30},
}

rows = []
for name in ['Exp 0', 'Exp 1', 'Exp 2A', 'Exp 2B', 'Exp 3', 'Exp 4']:
    if name not in experiments:
        continue
    e = experiments[name]
    m = e['metrics']
    t = e['training']

    # Check pass/fail
    c = criteria[name]
    passed = True
    if 'delta_mse' in c and m['delta_mse'] > c['delta_mse']:
        passed = False
    if 'sign_accuracy' in c and m.get('sign_accuracy', 0) < c['sign_accuracy']:
        passed = False
    if 'scm_validity' in c and m.get('scm_validity', 0) < c['scm_validity']:
        passed = False
    if 'loss_reduction' in c and t.get('loss_reduction', 0) < c['loss_reduction']:
        passed = False

    rows.append([
        name,
        e['config']['num_features'],
        f"{m['delta_mse']:.4f}",
        f"{m.get('sign_accuracy', 'N/A'):.2%}" if isinstance(m.get('sign_accuracy'), (int, float)) else 'N/A',
        f"{m.get('scm_validity', 'N/A'):.2%}" if isinstance(m.get('scm_validity'), (int, float)) else 'N/A',
        f"{t['loss_reduction']:.1%}",
        f"{t['train_time_seconds']:.0f}",
        'PASS' if passed else 'FAIL',
    ])

# Print as formatted table
print(f"{'Experiment':<12} {'Feat':>4} {'Delta MSE':>10} {'Sign Acc':>10} {'SCM Valid':>10} {'Loss Red':>10} {'Time(s)':>8} {'Result':>8}")
print("-" * 80)
for row in rows:
    print(f"{row[0]:<12} {row[1]:>4} {row[2]:>10} {row[3]:>10} {row[4]:>10} {row[5]:>10} {row[6]:>8} {row[7]:>8}")

## 2. Training Loss Curves

Overlaid training loss curves for all experiments. Note the log scale on the y-axis to accommodate the wide range of loss values across experiments.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = {
    'Exp 0': '#2196F3', 'Exp 1': '#4CAF50', 'Exp 2A': '#FF9800',
    'Exp 2B': '#F44336', 'Exp 3': '#9C27B0', 'Exp 4': '#795548',
}

# Left: all experiments on log scale
ax = axes[0]
for name in ['Exp 0', 'Exp 1', 'Exp 2A', 'Exp 2B', 'Exp 3', 'Exp 4']:
    if name not in experiments:
        continue
    log = experiments[name]['training_log']
    epochs = [e['epoch'] for e in log]
    losses = [e['loss'] for e in log]
    ax.semilogy(epochs, losses, label=name, color=colors[name], alpha=0.8, linewidth=1.5)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Training Loss — All Experiments')
ax.legend()

# Right: fixed-SCM experiments (0, 1, 2A, 2B) zoomed in
ax = axes[1]
for name in ['Exp 0', 'Exp 1', 'Exp 2A', 'Exp 2B']:
    if name not in experiments:
        continue
    log = experiments[name]['training_log']
    epochs = [e['epoch'] for e in log]
    losses = [e['loss'] for e in log]
    ax.semilogy(epochs, losses, label=name, color=colors[name], alpha=0.8, linewidth=1.5)

ax.set_xlabel('Epoch')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Training Loss — Fixed-SCM Experiments (0-2)')
ax.legend()

plt.tight_layout()
plt.show()

## 3. SCM Validity Progression

SCM validity measures whether predicted counterfactuals actually flip the label when fed back through the original causal model. This is the gold-standard metric.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

exp_names = [n for n in ['Exp 0', 'Exp 1', 'Exp 2A', 'Exp 2B', 'Exp 3', 'Exp 4'] if n in experiments]
validity_vals = [experiments[n]['metrics'].get('scm_validity', 0) for n in exp_names]
threshold_vals = [criteria[n].get('scm_validity', 0) for n in exp_names]
bar_colors = [colors[n] for n in exp_names]

# Bar chart of SCM validity
ax = axes[0]
x = np.arange(len(exp_names))
bars = ax.bar(x, validity_vals, color=bar_colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.scatter(x, threshold_vals, marker='_', color='red', s=200, linewidths=3, zorder=5, label='Threshold')
ax.set_xticks(x)
ax.set_xticklabels(exp_names)
ax.set_ylabel('SCM Validity')
ax.set_title('SCM Validity by Experiment')
ax.set_ylim(0, 1.05)
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')
ax.legend()

# Add value labels on bars
for bar, val in zip(bars, validity_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')

# Delta MSE comparison
ax = axes[1]
mse_vals = [experiments[n]['metrics']['delta_mse'] for n in exp_names]
mse_thresh = [criteria[n].get('delta_mse', None) for n in exp_names]

bars = ax.bar(x, mse_vals, color=bar_colors, alpha=0.8, edgecolor='black', linewidth=0.5)
ax.scatter(x, [t for t in mse_thresh if t is not None],
           marker='_', color='red', s=200, linewidths=3, zorder=5, label='Threshold')
ax.set_xticks(x)
ax.set_xticklabels(exp_names)
ax.set_ylabel('Delta MSE')
ax.set_title('Delta MSE by Experiment')
ax.set_yscale('log')
ax.legend()

for bar, val in zip(bars, mse_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.3,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

## 4. Context Ablation (Experiments 3 & 4)

To verify in-context learning, we compare model performance with:
- **Correct context**: data from the same SCM
- **Wrong context**: data from a different SCM
- **No context**: zeroed-out context labels

A large gap between correct and wrong context indicates the model uses context for SCM identification.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ablation_exps = ['Exp 3', 'Exp 4']
for idx, name in enumerate(ablation_exps):
    ax = axes[idx]
    if name not in experiments or 'context_ablation' not in experiments[name]:
        ax.text(0.5, 0.5, f'{name}: No ablation data', ha='center', va='center', transform=ax.transAxes)
        continue

    abl = experiments[name]['context_ablation']
    conditions = ['Correct\nContext', 'Wrong\nContext', 'No\nContext']
    values = [abl['correct_context'], abl['wrong_context'], abl['no_context']]
    bar_cols = ['#4CAF50', '#F44336', '#9E9E9E']

    bars = ax.bar(conditions, values, color=bar_cols, alpha=0.8, edgecolor='black', linewidth=0.5)
    ax.set_ylabel('SCM Validity')
    ax.set_title(f'{name} — Context Ablation')
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Random chance')

    for bar, val in zip(bars, values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{val:.1%}', ha='center', va='bottom', fontsize=11, fontweight='bold')

    gap = abl['correct_context'] - abl['wrong_context']
    ax.set_xlabel(f'Ablation gap: {gap:.1%}')
    ax.legend()

plt.tight_layout()
plt.show()

## 5. Feature Scaling Analysis (Exp 2A vs 2B)

How do metrics degrade as we scale from 5 to 10 features within the same fixed SCM?

In [ ]:
if 'Exp 2A' in experiments and 'Exp 2B' in experiments:
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    metrics_to_compare = [
        ('delta_mse', 'Delta MSE', False),
        ('scm_validity', 'SCM Validity', True),
        ('sign_accuracy', 'Sign Accuracy', True),
    ]

    for idx, (metric, label, higher_better) in enumerate(metrics_to_compare):
        ax = axes[idx]
        vals = []
        for name in ['Exp 2A', 'Exp 2B']:
            v = experiments[name]['metrics'].get(metric, 0)
            vals.append(v)

        feat_counts = [5, 10]
        bars = ax.bar(['5 features\n(Exp 2A)', '10 features\n(Exp 2B)'], vals,
                      color=[colors['Exp 2A'], colors['Exp 2B']], alpha=0.8,
                      edgecolor='black', linewidth=0.5)

        for bar, val in zip(bars, vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01 * max(vals),
                    f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

        ax.set_ylabel(label)
        ax.set_title(f'{label}: 5 vs 10 Features')

    plt.tight_layout()
    plt.show()

    # Print comparison
    print("Feature Scaling Summary:")
    print(f"  5 features -> 10 features:")
    for metric in ['delta_mse', 'scm_validity', 'sign_accuracy']:
        v2a = experiments['Exp 2A']['metrics'].get(metric, 'N/A')
        v2b = experiments['Exp 2B']['metrics'].get(metric, 'N/A')
        if isinstance(v2a, float) and isinstance(v2b, float):
            change = (v2b - v2a) / abs(v2a) * 100 if v2a != 0 else 0
            print(f"  {metric}: {v2a:.4f} -> {v2b:.4f} ({change:+.1f}%)")
else:
    print("Exp 2A or 2B not available")

## 6. Loss Reduction and Convergence Speed

How quickly and how far each experiment converged.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss reduction
ax = axes[0]
names = [n for n in ['Exp 0', 'Exp 1', 'Exp 2A', 'Exp 2B', 'Exp 3', 'Exp 4'] if n in experiments]
reductions = [experiments[n]['training']['loss_reduction'] for n in names]
bar_cols = [colors[n] for n in names]

bars = ax.bar(names, reductions, color=bar_cols, alpha=0.8, edgecolor='black', linewidth=0.5)
for bar, val in zip(bars, reductions):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.1%}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax.set_ylabel('Loss Reduction')
ax.set_title('Total Loss Reduction (initial -> final)')
ax.set_ylim(0, 1.15)

# Initial vs final loss
ax = axes[1]
x = np.arange(len(names))
width = 0.35
initial = [experiments[n]['training']['initial_loss'] for n in names]
final = [experiments[n]['training']['final_loss'] for n in names]

ax.bar(x - width/2, initial, width, label='Initial Loss', color='lightcoral', edgecolor='black', linewidth=0.5)
ax.bar(x + width/2, final, width, label='Final Loss', color='lightgreen', edgecolor='black', linewidth=0.5)
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.set_ylabel('Loss')
ax.set_title('Initial vs Final Loss')
ax.set_yscale('log')
ax.legend()

plt.tight_layout()
plt.show()

## 7. Conclusions

### What passed
- **Exp 0** (Linear): Near-perfect convergence (MSE ~1e-7). Pipeline is correct.
- **Exp 1** (Nonlinear, 3f): Strong results — MSE 0.003, 100% sign accuracy, 96% SCM validity.
- **Exp 2A** (5 features): Scales well — MSE 0.003, 100% sign accuracy, 97% SCM validity.
- **Exp 2B** (10 features): Still excellent — MSE 0.003, 100% sign accuracy, 99% SCM validity.
- **Exp 3** (SCM family, ICL): MSE 0.71, 97% SCM validity, 35% ablation gap — **strong in-context learning demonstrated**.

### What partially failed
- **Exp 4** (Diverse SCMs): SCM validity 51% (PASS, above 30% threshold), but MSE 2.22 (FAIL, above 1.0 threshold) and ablation gap 0% (FAIL, below 15% threshold). The model generalizes partially but does **not** perform in-context causal inference on unseen SCM structures.

### Key findings
1. **Fixed-SCM overfitting works perfectly** (Exp 0-2B): The transformer can learn to predict exact counterfactual deltas for a fixed causal model, even with 10 features.
2. **In-context SCM identification works** (Exp 3): With a fixed family of 10 SCMs, the model learns to use context data to identify which SCM generated the data and predict correct counterfactuals. The 35% ablation gap confirms genuine in-context learning.
3. **Diverse SCM generalization is the frontier** (Exp 4): When SCM structure varies randomly per batch, the model achieves above-chance validity but cannot use context for SCM identification (zero ablation gap). This suggests the problem requires either more context data, a larger model, or architectural changes to handle structural variation.

### Next steps
- Try larger models / longer training for Exp 4
- Experiment with curriculum learning (start with similar SCMs, gradually diversify)
- Consider providing structural hints (e.g., graph topology) as additional input
- Investigate whether direct x_cf prediction (instead of delta prediction) helps with diverse SCMs